# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 8: Advanced LLM Agents</font>

# <font color="#003660">Reminder: Simple LLM Agents</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how LLM agents are build with LangChain and LangGraph. <br>
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [LangChain Academy](https://academy.langchain.com/)
* [Introduction to LangChain Agents](https://github.com/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)
* [LangChain Docs (Python)](https://python.langchain.com/)

In [ ]:
!pip install -U wikipedia langchain langchain-community langchain-openai

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [ ]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [16]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)


Now we will have to download both models for this session. Run the code below.

In [ ]:
!ollama pull qwen3:8b # takes around a minute

## Answering Questions using LLMs

In [ ]:
from langchain_community.retrievers import WikipediaRetriever

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

Today we will use Qwen3, a reasoning LLM from Alibaba ([Yang et al., 2025](https://doi.org/10.48550/arXiv.2505.09388)). These are special LLMs trained to first reason step-by-step similar to Chain-of-Thought you learned last session. This training approach shifts LLMs away from fast, error-prone (System 1) responses toward more deliberate, reflective (System 2) reasoning about the task ([Li et a., 2025](https://doi.org/10.48550/arXiv.2502.17419)).

This is especially helpful when we want to do synthesis of information.

In [ ]:
@tool
def search_in_wikipedia(query: str) -> str:
    """Search Wikipedia for a given query."""
    retriever = WikipediaRetriever()
    docs = retriever.invoke(query)
    results = "\n\n-----\n\n".join([f"Document {i}:\n\nMetadata:\n-Title: {docs[i].metadata['title'].strip()}\n-Path: https://en.wikipedia.org/wiki/{docs[i].metadata['title'].strip().replace(' ', '_')}\n\nContent:\n{docs[i].metadata['summary'].strip()}" for i in range(len(docs))])

    return results

tools = [
    search_in_wikipedia
]

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable and helpful assistant that answers user queries.
You are able to retrieve and accurate information from Wikipedia."""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
react_agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
query = "Who is Daenerys Targaryen and who is her actress?"
messages = [HumanMessage(content=query)]
messages = react_agent.invoke({"messages": messages})["messages"]
for message in messages:
    print(message.pretty_repr())

## Build your own agent (A Blueprint)

In [ ]:
@tool
def your_tool_here(your_variables_here):
    # ToDo: implement your own tool
    pass

tools = [
    # your tools here
]

SYSTEM_PROMPT = """YOUR SYSTEM PROMTP HERE"""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
react_agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## Deep (Research) Agents

Deep (research) agents usually are able to plan (ToDo Lists), use the filesystem of the system executing them, and can also leverage subagents for different tasks ([Wang et al., 2025](https://doi.org/10.18653/v1/2025.findings-acl.259); [Zhang et al. 2024](https://doi.org/10.48550/arXiv.2405.16510); [Zhang et al., 2025](https://doi.org/10.48550/arXiv.2508.12752))

Implemented in LangChain (LangGraph) there is a `deepagents` library that you can use when you wnat to:
- Handle complex, multi-step tasks that require planning and decomposition
- Manage large amounts of context through file system tools
- Delegate work to specialized subagents for context isolation
- Persist memory across conversations and threads
- For simpler use cases, consider using LangChain’s create_agent or building a custom LangGraph workflow.

![](imgs/deep_agent.png)

(Source: [LangChain 2025](https://docs.langchain.com/oss/python/deepagents/middleware))

To develop this in LangChain and LangGraph, we will work with what is called [`states`](https://docs.langchain.com/oss/python/langgraph/graph-api).

As illustrated in the image below, multiple additional features need to extend the LLM agent and its state to provide appropriate deep agents.

![](https://raw.githubusercontent.com/langchain-ai/deep-agents-from-scratch/4d65d048419e77bd0b0da39f178c6d7cc32ecd99/notebooks/assets/agent_header.png)

(Source: [Langchain 2025](https://github.com/langchain-ai/deep-agents-from-scratch/blob/main/notebooks/4_full_agent.ipynb))

We will implement the `state` for the LLM agent planning with ToDo lists from scratch to get an understanding of `states`.